# Fase 5: Ingesta del Mega-Panel de Indicadores Económicos de Madrid

1. **Renta Neta Media de los Hogares:** En lugar de inferir un "marcador" de renta de 0 a 100, obtendremos el valor en EUROS reales, por barrio.
2. **Pensión Media Mensual:** De nuevo, valor exacto oficial en EUROS.
3. **Disponibilidad/Supervivencia (Rotación Comercial):** Ratio calculado en base a `Número de locales dados de alta abiertos` vs `cerrados`.

In [ ]:
import pandas as pd
import numpy as np

csv_path = '../data/processed/negocios_scored.csv'
df_base = pd.read_csv(csv_path)

def normalize_str(s):
    if pd.isna(s): return ''
    return str(s).upper().replace('Á', 'A').replace('É', 'E').replace('Í', 'I').replace('Ó', 'O').replace('Ú', 'U').strip()

df_base['barrio_norm'] = df_base['desc_barrio_local'].apply(normalize_str)

## 1. Carga Inteligente de Indicadores
Como el dataset es enorme (30MB+ en texto) y tiene formato "Largo" (Long), filtramos por las filas de indicadores que realmente nos importan.

In [ ]:
eco_path = '../data/datos_economicos_generales_madrid.csv'
print("Cargando mega-panel económico...")

# Omitimos líneas malas (errores de formato csv)
df_eco = pd.read_csv(eco_path, sep=';', encoding='utf-8', on_bad_lines='skip', low_memory=False)

indicadores_objetivo = [
    'Renta neta media anual de los hogares (Urban Audit)',
    'Pensión media mensual hombres',
    'Pensión media mensual mujeres',
    'Número de locales dados de alta abiertos',
    'Número de locales dados de alta cerrados'
]

# Filtrar sólo los indicadores útiles
df_eco_filt = df_eco[df_eco['indicador_completo'].isin(indicadores_objetivo)].copy()
df_eco_filt = df_eco_filt.dropna(subset=['barrio', 'valor_indicador'])

# Quedarnos con el dato más reciente (Último Año Informado) para cada barrio e indicador
df_eco_filt = df_eco_filt.sort_values(by='año', ascending=False)
df_eco_unico = df_eco_filt.drop_duplicates(subset=['barrio', 'indicador_completo'])

print(f"Extraídos {len(df_eco_unico)} valores para los indicadores listados.")

## 2. Pivotado a Columnas e Imputación
Pasamos de un dataframe vertical (un indicador por fila) a uno horizontal (cada barrio tiene sus columnas).

In [ ]:
# Pivot Table (Barrio x Indicador)
df_pivot = df_eco_unico.pivot(index='barrio', columns='indicador_completo', values='valor_indicador').reset_index()

# Limpiar formato de números en español (ej: "45.000,50" a 45000.5)
def clean_number(x):
    if pd.isna(x): return np.nan
    if isinstance(x, str):
        # Algunos valores tienen punto de millar y coma decimal.
        x = x.replace('.', '').replace(',', '.')
        try:
            return float(x)
        except: return np.nan
    return float(x)

for col in df_pivot.columns:
    if col != 'barrio':
        df_pivot[col] = df_pivot[col].apply(clean_number)

# Estandarización
df_pivot['barrio_norm'] = df_pivot['barrio'].apply(normalize_str)

# Construir features limpias finales
df_pivot['renta_neta_hogares'] = df_pivot['Renta neta media anual de los hogares (Urban Audit)']
df_pivot['pension_media_real'] = df_pivot[['Pensión media mensual hombres', 'Pensión media mensual mujeres']].mean(axis=1)

# Tasa de supervivencia / saturación comercial oficial
df_pivot['locales_abiertos'] = df_pivot['Número de locales dados de alta abiertos'].fillna(0)
df_pivot['locales_cerrados'] = df_pivot['Número de locales dados de alta cerrados'].fillna(0)
totales_censados = df_pivot['locales_abiertos'] + df_pivot['locales_cerrados']
# Proporción 0–1 (abiertos / total); generar_ranking_barrios y la UI pasan a % multiplicando ×100.
df_pivot['tasa_supervivencia_real'] = np.where(totales_censados > 0, 
                                               (df_pivot['locales_abiertos'] / totales_censados), 
                                               np.nan)

## 3. Merge y Enriquecimiento
Conectamos las nuevas métricas (Renta Neta Pura, Tasa de Supervivencia basad en el censo actual y Pensión media pura) al core del predictor.

In [ ]:
cols_to_merge = ['barrio_norm', 'renta_neta_hogares', 'pension_media_real', 'tasa_supervivencia_real', 'locales_abiertos', 'locales_cerrados']
df_base = pd.merge(df_base, df_pivot[cols_to_merge], on='barrio_norm', how='left')

# Imputación de nulos (en caso de algún barrio sin datos recientes, se pone la media de Madrid)
df_base['renta_neta_hogares'] = df_base['renta_neta_hogares'].fillna(df_base['renta_neta_hogares'].mean())
df_base['pension_media_real'] = df_base['pension_media_real'].fillna(df_base['pension_media_real'].mean())
df_base['tasa_supervivencia_real'] = df_base['tasa_supervivencia_real'].fillna(df_base['tasa_supervivencia_real'].mean())

df_base.drop(columns=['barrio_norm'], inplace=True, errors='ignore')

# Visualizamos la perfección
display(df_base[['desc_barrio_local', 'renta_neta_hogares', 'tasa_supervivencia_real']].head(3))

print("Volcando todos los datos empíricos. Preparado para producción.")
df_base.to_csv(csv_path, index=False)
print("¡Mega-Panel Ingestado con Éxito!")